In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

archive = Path(
    "/content/drive/MyDrive/AG-ReID.v2.zip"
)
local_archive = Path("/content/AGReIDv2.zip")

assert archive.exists(), f"Archive not found: {archive}"

shutil.copy2(archive, local_archive)
print(f"Copied {local_archive.stat().st_size / 1024**3:.2f} GiB")

In [ ]:
import zipfile
from pathlib import Path

extract_dir = Path("/content/datasets/AGReIDv2")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(local_archive) as zf:
    zf.extractall(extract_dir)

print("Extracted to:", extract_dir)

In [ ]:
%cd /content
import os
if not os.path.exists('/content/ViSA'):
    !git clone https://github.com/nam0403/ViSA
%cd /content/ViSA
!git fetch origin
!git checkout fix/colab-t4-training-optimizations
!git pull origin fix/colab-t4-training-optimizations


In [ ]:
from pathlib import Path
import urllib.request

save_dir = Path("/content/drive/MyDrive/ViSA/pretrained")
save_dir.mkdir(parents=True, exist_ok=True)

url = "https://dl.fbaipublicfiles.com/deit/deit_base_patch16_224-b5f2ef4d.pth"
save_path = save_dir / "deit_base_patch16_224-b5f2ef4d.pth"

if save_path.exists():
    print(f"File đã tồn tại: {save_path}")
    print(f"Size: {save_path.stat().st_size / 1024**2:.1f} MB")
else:
    urllib.request.urlretrieve(url, save_path)
    print(f"Download completed: {save_path}")
    print(f"Size: {save_path.stat().st_size / 1024**2:.1f} MB")

In [ ]:
from pathlib import Path

for p in [
    Path("/content/drive/MyDrive/ViSA/pretrained/deit_base_patch16_224-b5f2ef4d.pth"),
    Path("/content/datasets/AGReIDv2"),
    Path("/content/ViSA/configs/AGReIDv2/visa.yml"),
]:
    print(f"{'OK' if p.exists() else 'MISSING'}: {p}")

In [ ]:
from pathlib import Path

dataset_root = Path("/content/datasets/AGReIDv2")

for p in list(dataset_root.iterdir())[:20]:
    print("DIR " if p.is_dir() else "FILE", p.name)

In [ ]:
import torch

pth = "/content/drive/MyDrive/ViSA/pretrained/deit_base_patch16_224-b5f2ef4d.pth"
ckpt = torch.load(pth, map_location="cpu", weights_only=True)
state = ckpt.get("model", ckpt)

print("CLS:", state["cls_token"].shape)
print("Position:", state["pos_embed"].shape)
print("Patch projection:", state["patch_embed.proj.weight"].shape)

In [ ]:
from pathlib import Path

root = Path("/content/datasets/AGReIDv2/AG-ReID.v2")

print("Dataset root exists:", root.exists())

if root.exists():
    for p in list(root.iterdir())[:30]:
        print(("DIR " if p.is_dir() else "FILE"), p.name)

print("\n--- ViSA config ---")
%cd /content/ViSA
!grep -nE "PRETRAIN_PATH|PRETRAIN:|DATASETS|ROOT|SIZE_TRAIN|STRIDE_SIZE|IMS_PER_BATCH|MAX_EPOCH|OUTPUT_DIR|TEST" configs/AGReIDv2/visa.yml

In [ ]:
%cd /content/ViSA

# Kiểm tra file pretrained và code xử lý positional embedding
!ls -lh /content/drive/MyDrive/ViSA/pretrained/
!grep -n -A35 -B10 "resize_pos_embed" fastreid/modeling/backbones/vision_transformer_multiview_ViSA.py

# Kiểm tra cấu hình dataset mặc định
!grep -n -A15 -B5 "AG_ReID_v2" fastreid/data/datasets/agreidv2.py
!grep -n -A10 -B5 "DATASETS:" configs/AGReIDv2/visa.yml

In [ ]:
%cd /content/ViSA
!git status --short
!git checkout fix/colab-t4-training-optimizations
!git pull origin fix/colab-t4-training-optimizations


In [ ]:
!git log -1 --oneline
!ls configs/AGReIDv2/visa_colab.yml

In [ ]:
%cd /content/ViSA

!grep -nE "PRETRAIN_PATH|PRETRAIN:|IMS_PER_BATCH|OUTPUT_DIR|SIZE_TRAIN|STRIDE_SIZE|MAX_EPOCH|ROOT" \
    configs/AGReIDv2/visa_colab.yml

In [ ]:
%cd /content/ViSA
!python tools/train_net.py --help

In [ ]:
%cd /content/ViSA

from fastreid.config import get_cfg
from fastreid.modeling import build_model

cfg = get_cfg()
cfg.merge_from_file("configs/AGReIDv2/visa_colab.yml")
cfg.freeze()

print("Dataset:", cfg.DATASETS.ROOT)
print("Pretrained:", cfg.MODEL.BACKBONE.PRETRAIN_PATH)
print("Batch size:", cfg.SOLVER.IMS_PER_BATCH)
print("Output:", cfg.OUTPUT_DIR)

model = build_model(cfg)
print("Model initialized:", type(model).__name__)

In [ ]:
%cd /content/ViSA

from fastreid.data import build_reid_train_loader

train_loader = build_reid_train_loader(cfg)
print("Loader created:", type(train_loader).__name__)

batch = next(iter(train_loader))
print("Batch type:", type(batch))

if isinstance(batch, dict):
    for k, v in batch.items():
        if hasattr(v, "shape"):
            print(k, v.shape, v.dtype if hasattr(v, "dtype") else "")
        else:
            print(k, type(v).__name__)
else:
    print(batch)


In [ ]:
%cd /content/ViSA

!git log -1 --oneline
!grep -RniE "GradScaler|autocast|scaler.step|scaler.update" fastreid/engine
!grep -nE "PRETRAIN_PATH|IMS_PER_BATCH|OUTPUT_DIR|MAX_EPOCH" \
    configs/AGReIDv2/visa_colab.yml

In [ ]:
# ==============================================================================
# PREFLIGHT SMOKE TEST & BENCHMARK FOR TESLA T4
# ==============================================================================
import os, sys, time, torch
from pathlib import Path
%cd /content/ViSA

print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {gpu_name} ({total_mem:.2f} GiB)')
    assert 'T4' in gpu_name or 'Tesla' in gpu_name, f'Expected T4 GPU, got: {gpu_name}'
else:
    raise RuntimeError('CUDA is not available! Change Colab runtime to GPU (T4).')

from fastreid.config import get_cfg
from fastreid.engine import DefaultTrainer
from fastreid.utils.file_io import PathManager

cfg = get_cfg()
cfg.merge_from_file('configs/AGReIDv2/visa_colab.yml')
cfg.freeze()

print('Checking paths:')
assert Path(cfg.DATASETS.ROOT).exists(), f'Missing dataset: {cfg.DATASETS.ROOT}'
assert Path(cfg.MODEL.BACKBONE.PRETRAIN_PATH).exists(), f'Missing weights: {cfg.MODEL.BACKBONE.PRETRAIN_PATH}'
PathManager.mkdirs(cfg.OUTPUT_DIR)
print(f'  Dataset: OK ({cfg.DATASETS.ROOT})')
print(f'  Pretrained DeiT: OK ({cfg.MODEL.BACKBONE.PRETRAIN_PATH})')
print(f'  Output dir: OK ({cfg.OUTPUT_DIR})')

print('\nBuilding trainer, model, and DataLoader...')
trainer = DefaultTrainer(cfg)
print('Trainer constructed successfully. AMP enabled:', cfg.SOLVER.AMP.ENABLED)

print('\nRunning 5 warm-up iterations...')
for _ in range(5):
    trainer.run_step()
torch.cuda.synchronize()

print('Measuring steady-state performance over 10 iterations...')
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
for _ in range(10):
    trainer.run_step()
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

step_time = elapsed / 10
throughput = (cfg.SOLVER.IMS_PER_BATCH * 10) / elapsed
allocated_gib = torch.cuda.max_memory_allocated() / (1024**3)
reserved_gib = torch.cuda.max_memory_reserved() / (1024**3)

print(f'Average iteration time: {step_time:.4f} s / step ({throughput:.2f} images/s)')
print(f'Peak VRAM: {allocated_gib:.2f} GiB allocated, {reserved_gib:.2f} GiB reserved')
est_epoch_hrs = (len(trainer.data_loader) * step_time) / 3600
est_total_hrs = (len(trainer.data_loader) * step_time * cfg.SOLVER.MAX_EPOCH) / 3600
print(f'Iterations per epoch: {len(trainer.data_loader)}')
print(f'Estimated epoch time: {est_epoch_hrs:.2f} hours')
print(f'Estimated total 120-epoch time: {est_total_hrs:.1f} hours ({est_total_hrs/24:.2f} days)')

print('\nTesting checkpoint save and verification...')
test_ckpt_path = os.path.join(cfg.OUTPUT_DIR, 'preflight_test.pth')
trainer.checkpointer.save('preflight_test', test_metric=0.99)
assert os.path.exists(test_ckpt_path), f'Failed to create {test_ckpt_path}'
print('Preflight test checkpoint saved and verified successfully!')
print('\n>>> ALL PREFLIGHT CHECKS PASSED! Runtime is ready for full training. <<<')


In [ ]:
%cd /content/ViSA

!python -u tools/train_net.py --config-file configs/AGReIDv2/visa_colab.yml 2>&1 | tee /content/drive/MyDrive/ViSA/checkpoints/AGReIDv2/ViSA-Colab/train.log